In [ ]:
# ---- EXPERIMENT CONFIG: edit this cell for each run --------------------------
# Offline (RTX, no internet) runner for --model wignerd: Lee & Cho 2024, "3D Equivariant Pose Regression via Direct Wigner-D
# Harmonics Prediction" (arXiv 2411.00543). Paper recipe on Pascal3D+ with RenderForCNN synthetic data: ResNet-101 (ImageNet) ->
# Image2Sphere trunk (lmax 6, 512 sphere channels, 8 hidden SO(3) channels) -> 455 Wigner-D coefficients, frequency-domain MSE,
# SGD + Nesterov 0.9, lr 0.1 x0.1 every 30 epochs, batch 64, 80 epochs, no EMA, no gradient clipping; inference = argmax over the
# level-5 SO(3) HEALPix grid (2.36M rotations). Claimed: 8.6 deg mean of per-class medians. The weights w_l of the loss are not
# given in the paper (--wigner_loss_weights unit|uniform). Data: real PASCAL3D+ with warp + RenderForCNN renders from the RAM pack.
# The code comes from the repo snapshot dataset, the libraries from ga-research-offline-deps; W&B runs offline
# (gpu_pool/rtx_monitor.py replays the WANDB_SYNC log lines).
#
# Attach: syfry5suvzovvakmuj/pascal3d, pascal3d-ram-cache, pascal3d-synth-pack, ga-research-offline-deps and ONE repo snapshot.
RECIPE = ['--model', 'wignerd', '--encoder', 'resnet101', '--use_warp', '--use_synth', '--num_workers', '24', '--no-ema',
          '--batch_size', '64', '--optimizer', 'sgd', '--lr', '0.1', '--lr_scaling', 'none', '--lr_schedule', 'step',
          '--lr_step_epochs', '30', '--lr_step_gamma', '0.1', '--n_epochs', '80', '--grad_clip', '0', '--i2s_eval_rec_level', '5']
RUN_NAME = 'wignerd_pascal_synth'
EXTRA = []
SNAPSHOT_MARKER = '"wignerd"'   # the snapshot's config.py must contain it (stale-snapshot guard)
ARGS = RECIPE + EXTRA


In [ ]:
import glob, os, shutil, subprocess, tarfile

# The snapshot dataset is either auto-extracted by Kaggle (a GA-Research/ tree) or still a tar.gz.
if os.path.exists('/kaggle/working/GA-Research'):
    shutil.rmtree('/kaggle/working/GA-Research')
markers = glob.glob('/kaggle/input/**/pose3d/pose3d/config.py', recursive=True)
if markers:
    assert len(markers) == 1, f'attach exactly one repo snapshot, found {markers}'
    repo_root = markers[0][:markers[0].index('/pose3d/pose3d/config.py')]
    shutil.copytree(repo_root, '/kaggle/working/GA-Research')
else:
    tgzs = glob.glob('/kaggle/input/**/ga-research-*.tar.gz', recursive=True)
    assert len(tgzs) == 1, f'attach exactly one repo snapshot, found {tgzs}'
    with tarfile.open(tgzs[0]) as t:
        t.extractall('/kaggle/working')
print(sorted(os.listdir('/kaggle/working/GA-Research')))
config_src = open('/kaggle/working/GA-Research/pose3d/pose3d/config.py').read()
assert SNAPSHOT_MARKER in config_src, f'stale snapshot: {SNAPSHOT_MARKER!r} is not in config.py'

In [ ]:
# Libraries Kaggle lacks, from the offline wheels (no index, no deps: the image's torch stays).
deps = next(p for p in glob.glob('/kaggle/input/**/ga-research-offline-deps', recursive=True)
            + glob.glob('/kaggle/input/ga-research-offline-deps'))
wheels = sorted(glob.glob(os.path.join(deps, '*.whl')))
# GATr (the reference vector field) and einops: from any mounted *ga-research-gatr-deps* copy.
gatr_deps = glob.glob('/kaggle/input/**/ga-research-gatr-deps', recursive=True)
wheels += sorted(w for d in gatr_deps[:1] for w in glob.glob(os.path.join(d, '*.whl')))
print('wheels:', [os.path.basename(w) for w in wheels])
subprocess.run(['pip', 'install', '-q', '--no-index', '--no-deps', '--disable-pip-version-check', *wheels], check=True)

# torchvision's resnet101 DEFAULT weights, so the pretrained backbone needs no download.
ckpt_dir = os.path.expanduser('~/.cache/torch/hub/checkpoints')
os.makedirs(ckpt_dir, exist_ok=True)
shutil.copy(os.path.join(deps, 'resnet101-cd907fc2.pth'), os.path.join(ckpt_dir, 'resnet101-cd907fc2.pth'))

import importlib, sys
for mod in ['e3nn', 'healpy', 'image2sphere', 'clifford', 'opt_einsum']:
    importlib.import_module(mod)
if '--vector_field' in ARGS and ARGS[ARGS.index('--vector_field') + 1] == 'gatr':
    sys.path.insert(0, '/kaggle/working/GA-Research/pose3d')
    from pose3d.models.gatr_denoiser import _import_gatr
    print('GATr:', _import_gatr())   # fails here, not hours in, when the gatr-deps copy is missing
print('python', sys.version.split()[0], '| all offline deps import cleanly')

In [ ]:
# No internet: wandb writes local files only; the trainer also prints WANDB_SYNC lines per epoch.
os.environ['WANDB_MODE'] = 'offline'

In [ ]:
import sys, os
os.chdir('/kaggle/working/GA-Research/pose3d')   # the package is found from here, no install needed
PASCAL = next(p for p in ['/kaggle/input/datasets/syfry5suvzovvakmuj/pascal3d', '/kaggle/input/pascal3d']
              if os.path.exists(p))
cmd = [sys.executable, '-u', '-m', 'pose3d', f'--path_to_datasets={PASCAL}', '--platform=kaggle',
       f'--run_name={RUN_NAME}', *ARGS]
print(' '.join(cmd))
# check=True so a crash marks the Kaggle run as failed instead of passing silently.
subprocess.run(cmd, check=True, env={**os.environ})